# Benchmark 4: throughput vs cluster size
Same write load at N=3 vs N=5 nodes, showing the replication-cost tradeoff. Run `bench/04-cluster-size-throughput.sh` first.

In [ ]:
import sys
sys.path.insert(0, ".")
import matplotlib.pyplot as plt
from plot_style import load, agg


In [ ]:
df = load("cluster-size-throughput.csv")
df

In [ ]:
throughput_summary = agg(df, ["clusterSize"], "reqPerSec")

fig, ax = plt.subplots()
x = range(len(throughput_summary))
ax.bar(x, throughput_summary["mean"], yerr=throughput_summary["std"], capsize=5, color="tab:blue")
ax.set_xticks(list(x), throughput_summary["clusterSize"].astype(str))
ax.set_xlabel("cluster size (N)")
ax.set_ylabel("requests/sec")
ax.set_title(f"Throughput vs cluster size (mean ± stddev, n={df['run'].nunique()} runs)")
for i, v in enumerate(throughput_summary["mean"]):
    ax.text(i, v, f"{v:.0f}", ha="center", va="bottom")
plt.show()

In [ ]:
group_cols = ["clusterSize"]
latency_summary = agg(df, group_cols, "p50Ms")[group_cols]
for pct in ("p50Ms", "p95Ms", "p99Ms"):
    pct_summary = agg(df, group_cols, pct)
    latency_summary[f"{pct}_mean"] = pct_summary["mean"]
    latency_summary[f"{pct}_std"] = pct_summary["std"]

fig, ax = plt.subplots()
width = 0.25
x = range(len(latency_summary))
ax.bar([i - width for i in x], latency_summary["p50Ms_mean"], width, yerr=latency_summary["p50Ms_std"], capsize=3, label="p50")
ax.bar(x, latency_summary["p95Ms_mean"], width, yerr=latency_summary["p95Ms_std"], capsize=3, label="p95")
ax.bar([i + width for i in x], latency_summary["p99Ms_mean"], width, yerr=latency_summary["p99Ms_std"], capsize=3, label="p99")
ax.set_xticks(list(x), latency_summary["clusterSize"].astype(str))
ax.set_xlabel("cluster size (N)")
ax.set_ylabel("latency (ms)")
ax.set_title(f"Latency vs cluster size (mean ± stddev, n={df['run'].nunique()} runs)")
ax.legend()
plt.show()